# Colab Session: switchglobe-globev2-20260905-a100
Generated from colab-cli history log.

**Session Created**: 2026-09-04 19:56:12
- Endpoint: `gpu-a100-s-kkb-usc1f1-nsi4iyqj0j8v`
- Hardware: `A100`

*File Operation*: `upload` on `/content/switchglobe_globev2_source.zip`

*File Operation*: `upload` on `/content/phase12_checkpoints.tar.gz`

*File Operation*: `upload` on `/content/fast_checkpoints.tar.gz`

In [ ]:
"""Prepare a self-contained SwitchGLOBE checkout on a Colab runtime."""

from __future__ import annotations

import hashlib
from pathlib import Path
import shutil
import subprocess
import sys


CONTENT = Path("/content")
ROOT = CONTENT / "SwitchGLOBE_globev2"
CHECKPOINT_ROOT = CONTENT / "switchglobe_checkpoints"


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run(command: list[str], *, cwd: Path | None = None) -> None:
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)


def main() -> None:
    source_archive = CONTENT / "switchglobe_globev2_source.zip"
    exact_archive = CONTENT / "phase12_checkpoints.tar.gz"
    fast_archive = CONTENT / "fast_checkpoints.tar.gz"
    for path in (source_archive, exact_archive, fast_archive):
        if not path.is_file():
            raise FileNotFoundError(path)

    shutil.rmtree(ROOT, ignore_errors=True)
    shutil.rmtree(CHECKPOINT_ROOT, ignore_errors=True)
    ROOT.mkdir(parents=True)
    (CHECKPOINT_ROOT / "phase12").mkdir(parents=True)
    (CHECKPOINT_ROOT / "fast").mkdir(parents=True)

    run(["unzip", "-q", str(source_archive), "-d", str(ROOT)])
    run([
        "tar", "-xzf", str(exact_archive), "-C",
        str(CHECKPOINT_ROOT / "phase12"), "--strip-components=2",
    ])
    run([
        "tar", "-xzf", str(fast_archive), "-C",
        str(CHECKPOINT_ROOT / "fast"), "--strip-components=1",
    ])

    # Colab already supplies CUDA-enabled torch/numpy/matplotlib. Install only
    # the small missing runtime packages, then install this checkout without
    # changing the pre-provisioned torch build.
    run([sys.executable, "-m", "pip", "install", "-q", "gymnasium>=1.0", "pyyaml>=6.0"])
    run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=ROOT)

    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU:", gpu)
    print("Python:", sys.version.replace("\n", " "))
    for path in (source_archive, exact_archive, fast_archive):
        print(f"SHA256 {path.name}: {sha256(path)}")
    print("Exact checkpoint files:", len(list((CHECKPOINT_ROOT / "phase12").glob("seed_*/*.pt"))))
    print("Fast checkpoint files:", len(list((CHECKPOINT_ROOT / "fast").glob("seed_*/*.pt"))))


if __name__ == "__main__":
    main()


+ unzip -q /content/switchglobe_globev2_source.zip -d /content/SwitchGLOBE_globev2


+ tar -xzf /content/phase12_checkpoints.tar.gz -C /content/switchglobe_checkpoints/phase12 --strip-components=2


+ tar -xzf /content/fast_checkpoints.tar.gz -C /content/switchglobe_checkpoints/fast --strip-components=1


+ /usr/bin/python3 -m pip install -q gymnasium>=1.0 pyyaml>=6.0


+ /usr/bin/python3 -m pip install -q --no-deps -e .


GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
SHA256 switchglobe_globev2_source.zip: b0aa2e386c7339e9d1f0b0097bc5bf9eee1247ee86084466da0eeb18d64041fe
SHA256 phase12_checkpoints.tar.gz: 6e8c4edc9f435db8f4458c191e82bd41983e7976500e3dd02e1b39601a5a262c
SHA256 fast_checkpoints.tar.gz: 83e03ba9536a7cf2031d69d4456dfd53a05d84cf40cffba939d3f29dfc5ae324
Exact checkpoint files: 10
Fast checkpoint files: 10


*File Operation*: `ls` on `/content/switchglobe_checkpoints/phase12/seed_42`

*File Operation*: `ls` on `/content/switchglobe_globev2_a100_20260905`

In [ ]:
"""Run and package the current-commit SwitchGLOBE A100 latency benchmark."""

from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time

import torch


CONTENT = Path("/content")
ROOT = CONTENT / "SwitchGLOBE_globev2"
CHECKPOINT_ROOT = CONTENT / "switchglobe_checkpoints"
OUTPUT = CONTENT / "switchglobe_globev2_a100_20260905"
ARCHIVE = CONTENT / "switchglobe_globev2_a100_20260905.zip"
SOURCE_COMMIT = "92d17df3f4451e0412858a9927a898a2696023b3"


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def main() -> None:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    started = time.time()
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else "NONE"
    hardware = {
        "requested_accelerator": "A100",
        "cuda_available": gpu_available,
        "gpu_name": gpu_name,
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "python": sys.version,
        "platform": platform.platform(),
        "source_git_commit": SOURCE_COMMIT,
        "torch_num_threads": 1,
        "batch_size": 1,
    }
    (OUTPUT / "hardware.json").write_text(
        json.dumps(hardware, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if not gpu_available or "A100" not in gpu_name.upper():
        raise RuntimeError(f"A100 verification failed: cuda={gpu_available}, name={gpu_name!r}")

    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)
    command = [
        sys.executable, "-m", "implementations.lite_globe.run_latency_benchmark",
        "--checkpoint-dir", str(CHECKPOINT_ROOT / "phase12"),
        "--fast-checkpoint-dir", str(CHECKPOINT_ROOT / "fast"),
        "--output-dir", str(OUTPUT),
        "--include-fast", "--include-early-exit",
        "--warmup", "50", "--repeats", "2000", "--zip-results",
    ]
    completed = subprocess.run(
        command, cwd=ROOT, text=True, capture_output=True,
    )
    (OUTPUT / "benchmark_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (OUTPUT / "benchmark_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode != 0:
        raise subprocess.CalledProcessError(
            completed.returncode, command, completed.stdout, completed.stderr
        )

    manifest_path = OUTPUT / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest.update({
        "execution_backend": "google_colab_cli",
        "requested_accelerator": "A100",
        "verified_gpu_name": gpu_name,
        "source_git_commit": SOURCE_COMMIT,
        "torch_num_threads": 1,
        "batch_size": 1,
        "elapsed_seconds": time.time() - started,
    })
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        text=True, stdout=(OUTPUT / "pip_freeze.txt").open("w", encoding="utf-8"),
        check=True,
    )

    if ARCHIVE.exists():
        ARCHIVE.unlink()
    shutil.make_archive(str(ARCHIVE.with_suffix("")), "zip", root_dir=OUTPUT)
    print(json.dumps({
        "complete": True,
        "archive": str(ARCHIVE),
        "archive_bytes": ARCHIVE.stat().st_size,
        "archive_sha256": sha256(ARCHIVE),
        "gpu_name": gpu_name,
        "elapsed_seconds": time.time() - started,
    }, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


{
  "complete": true,
  "archive": "/content/switchglobe_globev2_a100_20260905.zip",
  "archive_bytes": 194399,
  "archive_sha256": "07bbd3abb9991ed6293d8326926ea2f154453db47cc76c1923da709688f12b41",
  "gpu_name": "NVIDIA A100-SXM4-40GB",
  "elapsed_seconds": 775.9402148723602
}


*File Operation*: `download` on `/content/switchglobe_globev2_a100_20260905.zip`